Trying to recreate the preprocessing process of the core paper experiment.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Change these paths if your CSV files are somewhere else
DATA_PATH = Path("Raw_Dataset/diabetic_data.csv")
MAP_PATH = Path("Raw_Dataset/IDS_mapping.csv")

# Important:
# keep_default_na=False keeps the string "None" in A1Cresult.
# In this dataset, "None" means "test was not measured", not a missing Python value.
df = pd.read_csv(
    DATA_PATH,
    na_values=["?"],
    keep_default_na=False,
    low_memory=False
)

ids_mapping = pd.read_csv(MAP_PATH, keep_default_na=False)

print(df.shape)
df.head()

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [2]:
clean = df.copy()

# 1. Drop columns the paper did not use because of too many missing values
clean = clean.drop(columns=["weight", "payer_code"])

# 2. Keep medical_specialty, but mark missing values clearly
clean["medical_specialty"] = clean["medical_specialty"].fillna("Missing")
clean["race"] = clean["race"].fillna("Missing")

# 3. Remove invalid gender rows
# There are only a few, so this is safe.
clean = clean[clean["gender"] != "Unknown/Invalid"].copy()

# 4. Remove death / hospice discharge cases
# From IDS_mapping:
# 11 = Expired
# 13 = Hospice / home
# 14 = Hospice / medical facility
# 19, 20, 21 = expired / hospice related
death_hospice_ids = [11, 13, 14, 19, 20, 21]

clean = clean[
    ~clean["discharge_disposition_id"].isin(death_hospice_ids)
].copy()

# 5. Keep only the first encounter for each patient
clean = (
    clean
    .sort_values(["patient_nbr", "encounter_id"])
    .drop_duplicates(subset="patient_nbr", keep="first")
)

# 6. Define binary 30-day readmission outcome
# 1 = readmitted within 30 days
# 0 = not readmitted within 30 days
clean["readmitted_30"] = (clean["readmitted"] == "<30").astype(int)

print(clean.shape)
print(clean["readmitted_30"].value_counts())
print(clean["readmitted_30"].mean())

(69987, 49)
readmitted_30
0    63702
1     6285
Name: count, dtype: int64
0.08980239187277637


In [3]:
def make_a1c_group(row):
    if row["A1Cresult"] == "None":
        return "Not measured"
    elif row["A1Cresult"] == ">8" and row["change"] == "Ch":
        return "High, medication changed"
    elif row["A1Cresult"] == ">8" and row["change"] == "No":
        return "High, medication not changed"
    else:
        # This includes "Norm" and ">7"
        # The paper's table has a "normal result" group, but the public dataset also has ">7".
        # We can document this later.
        return "Normal/near-normal measured"


def diagnosis_group(code):
    code = str(code).strip()

    if code == "" or code.lower() == "nan":
        return "Other"

    # ICD codes beginning with V or E are special codes.
    # The paper groups these into Other.
    if code.startswith("V") or code.startswith("E"):
        return "Other"

    try:
        num = float(code)
    except ValueError:
        return "Other"

    if (390 <= num <= 459) or num == 785:
        return "Circulatory"
    elif (460 <= num <= 519) or num == 786:
        return "Respiratory"
    elif (520 <= num <= 579) or num == 787:
        return "Digestive"
    elif code.startswith("250"):
        return "Diabetes"
    elif 800 <= num <= 999:
        return "Injury"
    elif 710 <= num <= 739:
        return "Musculoskeletal"
    elif (580 <= num <= 629) or num == 788:
        return "Genitourinary"
    elif 140 <= num <= 239:
        return "Neoplasms"
    else:
        return "Other"


def age_group(age):
    # age looks like "[50-60)"
    left = int(age.strip("[]()").split("-")[0])

    if left < 30:
        return "<30"
    elif left < 60:
        return "30-60"
    else:
        return "60+"


def specialty_group(s):
    if s == "Missing":
        return "Missing"
    elif s == "InternalMedicine":
        return "Internal Medicine"
    elif s == "Cardiology":
        return "Cardiology"
    elif s == "Family/GeneralPractice":
        return "Family/General Practice"
    elif s.startswith("Surgery"):
        return "Surgery"
    else:
        return "Other"


def race_group(r):
    if r == "AfricanAmerican":
        return "African American"
    elif r == "Caucasian":
        return "Caucasian"
    elif r == "Missing":
        return "Missing"
    else:
        return "Other"


clean["a1c_group"] = clean.apply(make_a1c_group, axis=1)
clean["diag1_group"] = clean["diag_1"].apply(diagnosis_group)
clean["age_group"] = clean["age"].apply(age_group)
clean["specialty_group"] = clean["medical_specialty"].apply(specialty_group)
clean["race_group"] = clean["race"].apply(race_group)

clean["admission_source_group"] = np.select(
    [
        clean["admission_source_id"].eq(7),
        clean["admission_source_id"].isin([1, 2, 3])
    ],
    [
        "Emergency",
        "Referral"
    ],
    default="Other"
)

clean["discharge_group"] = np.where(
    clean["discharge_disposition_id"].eq(1),
    "Home",
    "Other"
)

In [4]:
columns_to_check = [
    "a1c_group",
    "diag1_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "specialty_group",
    "race_group"
]

for col in columns_to_check:
    print("\n", col)
    print(clean[col].value_counts())


 a1c_group
a1c_group
Not measured                    57141
Normal/near-normal measured      6607
High, medication changed         4058
High, medication not changed     2181
Name: count, dtype: int64

 diag1_group
diag1_group
Circulatory        21389
Other              12134
Respiratory         9491
Digestive           6488
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3441
Neoplasms           2538
Name: count, dtype: int64

 age_group
age_group
60+      46308
30-60    21871
<30       1808
Name: count, dtype: int64

 admission_source_group
admission_source_group
Emergency    37271
Referral     22792
Other         9924
Name: count, dtype: int64

 discharge_group
discharge_group
Home     44320
Other    25667
Name: count, dtype: int64

 specialty_group
specialty_group
Missing                    33652
Other                      12824
Internal Medicine          10641
Family/General Practice     4978
Cardiology                  4207
Surgery   

In [5]:
Path("processed").mkdir(exist_ok=True)

clean.to_csv("processed/diabetes_preprocessed_stage1.csv", index=False)

print("Saved cleaned dataset.")
print(clean.shape)

Saved cleaned dataset.
(69987, 56)
